[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haraldsDev/pandemic-diary-map/blob/main/annotation-process/05-llm-batch-notebook.ipynb)


In [14]:
# =============================================================================
# CONFIGURATION + MODEL SELECTION (UI-DRIVEN, NO HARDCODED MODEL)
# =============================================================================

import os
import time
import csv
import requests
from datetime import datetime
from zoneinfo import ZoneInfo
from datetime import datetime

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import drive


TIMEZONE = ZoneInfo("Europe/Riga")

# ------------------------------------------------------------------
# GOOGLE DRIVE
# ------------------------------------------------------------------
drive.mount("/content/drive")

# ------------------------------------------------------------------
# API KEY UI (COPY–PASTE)
# ------------------------------------------------------------------
api_key_widget = widgets.Password(
    description="OpenRouter API key:",
    placeholder="paste API key here",
    layout=widgets.Layout(width="450px"),
    style={"description_width": "120px"}
)
display(api_key_widget)

# Folder on the mounted Drive. Paste your own folder path.
# No other input files are required. Copy these two from this repository into that folder:
# annotation-process/28-input-csv-for-LLM-to-annotate.csv
# annotation-process/04-llm-annotation-prompt.md
project_root_widget = widgets.Text(
    description="Drive folder:",
    value="/content/drive/MyDrive/Haralds_PhD/2-gads/SH-Conf-Minho-SEP-2026/LLM-classification-notebook",
    layout=widgets.Layout(width="900px"),
    style={"description_width": "120px"},
)
display(project_root_widget)

# ------------------------------------------------------------------
# TEST MODE
# ------------------------------------------------------------------
TEST_MODE = False          # True = run only first N diary entries
TEST_LIMIT = 10           # how many diary entries in test mode

# ------------------------------------------------------------------
# API
# ------------------------------------------------------------------
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

# ------------------------------------------------------------------
# RUNTIME
# ------------------------------------------------------------------
SLEEP_SECONDS = 0.3          # rate limiting between API calls
TIMEZONE = ZoneInfo("Europe/Riga")

# ------------------------------------------------------------------
# MODEL SELECTION UI
# ------------------------------------------------------------------
model_dropdown = widgets.Dropdown(
    options=[
        ("Gemini 2.5 Flash (default)", "google/gemini-2.5-flash"),
        ("Gemini 2.5 Pro", "google/gemini-2.5-pro"),
        ("GPT-5", "openai/gpt-5"),
        ("Claude Sonnet", "anthropic/claude-sonnet-4"),
        ("Mistral Medium 3.1", "mistralai/mistral-medium-3.1"),
    ],
    value="google/gemini-2.5-flash",
    description="Modelis:",
    layout=widgets.Layout(width="450px"),
    style={"description_width": "120px"}
)

model_custom = widgets.Text(
    description="Cits modelis:",
    placeholder="piem., openai/gpt-4.1-mini",
    layout=widgets.Layout(width="450px"),
    style={"description_width": "120px"}
)

display(widgets.VBox([
    model_dropdown,
    model_custom
]))

# ------------------------------------------------------------------
# HELPER: resolve chosen model at runtime
# ------------------------------------------------------------------
def get_chosen_model() -> str:
    """
    Returns the model selected in the UI.
    Custom model (if provided) overrides dropdown.
    """
    return (
        model_custom.value.strip()
        if model_custom.value.strip()
        else model_dropdown.value
    )

# =============================================================================
# MAIN ANNOTATION RUNNER
# =============================================================================

def run_llm_annotation():
    # ------------------------------
    # READ API KEY FROM UI
    # ------------------------------
    API_KEY = api_key_widget.value.strip()
    if not API_KEY:
        raise ValueError("OpenRouter API key not provided.")

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }

    project_root = project_root_widget.value.strip().rstrip("/")
    if not project_root:
        raise ValueError("Google Drive folder not provided.")

    input_csv = os.path.join(project_root, "28-input-csv-for-LLM-to-annotate.csv")
    output_dir = os.path.join(project_root, "rezultati")
    os.makedirs(output_dir, exist_ok=True)

    model_id = get_chosen_model()

    # Output names use the model chosen when Run is clicked.
    run_type = "test" if TEST_MODE else "full"
    model_safe = model_id.replace("/", "-")
    timestamp = datetime.now(TIMEZONE).strftime("%Y-%m-%d_%H-%M")

    ANNOTATIONS_CSV = os.path.join(
        output_dir,
        f"{run_type}__{model_safe}__{timestamp}.csv"
    )

    PROGRESS_CSV = os.path.join(
        output_dir,
        f"llm_progress__{model_safe}__{timestamp}.csv"
    )

    ERROR_LOG = os.path.join(output_dir, "llm_errors.log")

    # ------------------------------
    # LOAD INPUT CSV
    # ------------------------------
    df = pd.read_csv(input_csv)

    if TEST_MODE:
        df = df.head(TEST_LIMIT)

    # ------------------------------
    # PREPARE OUTPUT FILES
    # ------------------------------
    with open(ANNOTATIONS_CSV, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "annotation_id",
            "author_there_now_1_0",
            "confidence_0_100",
            "annotation_notes",
            "model",
            "timestamp"
        ])

    with open(PROGRESS_CSV, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["timestamp", "diary_id", "status"])

    # ------------------------------
    # LOAD PROMPT
    # ------------------------------
    prompt_path = os.path.join(project_root, "04-llm-annotation-prompt.md")
    with open(prompt_path, "r", encoding="utf-8") as f:
        annotation_prompt = f.read()

    # ------------------------------
    # PROCESS ENTRIES
    # ------------------------------
    for idx, row in df.iterrows():
        diary_id = row["diary_id"]
        entry_text = row["entry_text"]
        geolocation_tasks = row["geolocation_tasks"]

        clear_output(wait=True)
        print(f"Processing diary_id={diary_id} ({idx + 1}/{len(df)})")

        payload = {
            "model": model_id,
            "messages": [
                {
                    "role": "user",
                    "content": f"""
{annotation_prompt}

--------------------
DIARY ENTRY:
{entry_text}

--------------------
GEOLOCATION TASKS:
{geolocation_tasks}
""".strip()
                }
            ],
            "max_tokens": 1600
        }

        try:
            response = requests.post(
                OPENROUTER_URL,
                headers=headers,
                json=payload,
                timeout=120
            )
            response.raise_for_status()
            output_text = response.json()["choices"][0]["message"]["content"]

            with open(ANNOTATIONS_CSV, "a", encoding="utf-8-sig", newline="") as f:
                f.write(output_text.strip() + "\n")

            status = "OK"

        except Exception as e:
            status = "ERROR"
            with open(ERROR_LOG, "a", encoding="utf-8") as f:
                f.write(f"{datetime.now(TIMEZONE)} | diary_id={diary_id} | {e}\n")

        with open(PROGRESS_CSV, "a", encoding="utf-8-sig", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                datetime.now(TIMEZONE).strftime("%Y-%m-%d %H:%M:%S"),
                diary_id,
                status
            ])

        time.sleep(SLEEP_SECONDS)

    print("✅ Annotation run completed.")

# =============================================================================
# RUN BUTTON
# =============================================================================

run_button = widgets.Button(
    description="Run LLM annotation",
    button_style="success"
)

display(run_button)

def on_run_clicked(b):
    run_llm_annotation()

run_button.on_click(on_run_clicked)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Password(description='OpenRouter API key:', layout=Layout(width='450px'), placeholder='paste API key here', st…

Button(button_style='success', description='Run LLM annotation', style=ButtonStyle())